# Câu 3 – Toán Tử LBP (Local Binary Patterns)
## Xử Lý Ảnh Số

---

**Đề bài:**

Cho 1 ảnh màu I kích thước n × m, chuyển đổi ảnh I thành ảnh xám. Dùng toán tử LBP (Local Binary Patterns) để biến đổi ảnh I theo các yêu cầu sau:
- Neighbors **P = 8**, R = 1 và R = 2
- Neighbors **P = 16**, R = 2 và R = 3
- Neighbors **P = 24**, R = 3

**Ghi chú:** Trường hợp P = 16 (hoặc P = 24) thì tách chuỗi nhị phân thành 2 phần (hoặc 3 phần). Tức là mỗi phần chuỗi có 8 bits. Sau đó, chỉ lấy phần có **giá trị lớn nhất** gán cho pixel đang xét (center pixel).

---

| STT | P | R | Số nhóm | Kết quả |
|:---:|:---:|:---:|:---:|:---:|
| 1 | 8 | 1 | 1 | $V_1$ |
| 2 | 8 | 2 | 1 | $V_1$ |
| 3 | 16 | 2 | 2 | $\max(V_1, V_2)$ |
| 4 | 16 | 3 | 2 | $\max(V_1, V_2)$ |
| 5 | 24 | 3 | 3 | $\max(V_1, V_2, V_3)$ |

---
## Bước 1 – Import thư viện

In [1]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

---
## Bước 2 – Đọc ảnh và chuyển sang ảnh xám

Đề bài yêu cầu từ ảnh màu I, chuyển sang ảnh xám trước rồi mới áp dụng LBP.

**Công thức chuyển đổi chuẩn (ITU-R BT.601):**
$$\text{Gray} = 0.299 \times R + 0.587 \times G + 0.114 \times B$$

Thư viện PIL tự động áp dụng công thức này khi gọi `.convert('L')`.

**Lưu ý về pixel biên (border):** Khi áp dụng LBP với bán kính R, các pixel cách mép ảnh dưới R pixel **không có đủ lân cận** để tính → gán giá trị 0.

$$\text{Số hàng/cột biên bỏ qua} = \lceil R \rceil$$

---
## Bước 3 – Các hàm tính LBP

### 3.1. Hàm nội suy song tuyến (Bilinear Interpolation)

Tọa độ điểm lân cận thứ $p$ trên vòng tròn bán kính $R$:
$$x_p = cx + R \cdot \cos\!\left(\frac{2\pi p}{P}\right) \qquad y_p = cy - R \cdot \sin\!\left(\frac{2\pi p}{P}\right)$$

*(Dấu trừ ở $y_p$ vì trục Y ảnh đi từ trên xuống dưới)*

Khi tọa độ là số thực (không phải nguyên), ta nội suy từ 4 pixel góc gần nhất:

$$g_p = (1-dy)(1-dx)\cdot f_{00} + (1-dy)\cdot dx\cdot f_{01} + dy\cdot(1-dx)\cdot f_{10} + dy\cdot dx\cdot f_{11}$$

trong đó $dx = x_p - \lfloor x_p \rfloor$, $\; dy = y_p - \lfloor y_p \rfloor$

### 3.2. Tạo chuỗi bit và tính giá trị thập phân

$$\text{bit}_p = \begin{cases} 1 & \text{nếu } g_p \geq g_c \\ 0 & \text{nếu } g_p < g_c \end{cases}$$

Quy đổi từng nhóm 8-bit sang thập phân (bit_0 = LSB):
$$V = \sum_{i=0}^{7} \text{bit}_i \times 2^i \in [0,\; 255]$$

### 3.3. Xử lý đặc biệt theo đề bài:
- **P = 8** → 1 nhóm → $\text{LBP} = V_1$
- **P = 16** → 2 nhóm → $\text{LBP} = \max(V_1, V_2)$
- **P = 24** → 3 nhóm → $\text{LBP} = \max(V_1, V_2, V_3)$

In [2]:
def noi_suy_song_tuyen(anh, y, x):
    """
    Nội suy song tuyến tại tọa độ thực (x, y).
    Ước lượng giá trị pixel từ 4 pixel góc gần nhất.

    Công thức:
    gp = (1-dy)(1-dx)*f00 + (1-dy)*dx*f01 + dy*(1-dx)*f10 + dy*dx*f11
    """
    rows, cols = anh.shape

    x0 = int(np.floor(x));  x1 = x0 + 1
    y0 = int(np.floor(y));  y1 = y0 + 1

    # Giữ trong biên ảnh
    x0 = np.clip(x0, 0, cols-1);  x1 = np.clip(x1, 0, cols-1)
    y0 = np.clip(y0, 0, rows-1);  y1 = np.clip(y1, 0, rows-1)

    dx = x - np.floor(x)   # phần lẻ theo chiều ngang
    dy = y - np.floor(y)   # phần lẻ theo chiều dọc

    f00 = anh[y0, x0]   # góc trái-trên
    f01 = anh[y0, x1]   # góc phải-trên
    f10 = anh[y1, x0]   # góc trái-dưới
    f11 = anh[y1, x1]   # góc phải-dưới

    return (1-dy)*(1-dx)*f00 + (1-dy)*dx*f01 \
         +    dy *(1-dx)*f10 +    dy *dx*f11


def tinh_lbp_mot_pixel(anh_xam, cy, cx, P, R):
    """
    Tính giá trị LBP cho 1 pixel trung tâm tại (cy, cx).
    Trả về giá trị thuộc [0, 255].

    Các bước:
      1. Lấy gc = giá trị pixel trung tâm
      2. Với mỗi lân cận p = 0..P-1:
         a. Tính tọa độ thực (xp, yp) trên vòng tròn bán kính R
         b. Nội suy song tuyến lấy gp
         c. So sánh gp với gc tạo bit nhị phân
      3. Chia chuỗi P bit thành nhóm 8-bit, quy đổi sang thập phân
      4. Lấy giá trị lớn nhất (MAX) trong các nhóm
    """
    gc = anh_xam[cy, cx]   # giá trị pixel trung tâm
    bits = []              # chuỗi bit nhị phân

    for p in range(P):
        # Bước 2a: tọa độ điểm lân cận thứ p trên vòng tròn
        goc = 2 * np.pi * p / P
        xp  = cx + R * np.cos(goc)
        yp  = cy - R * np.sin(goc)   # dấu trừ: trục Y ảnh ngược toán học

        # Bước 2b: nội suy song tuyến để lấy giá trị tại (xp, yp)
        gp = noi_suy_song_tuyen(anh_xam, yp, xp)

        # Bước 2c: tạo bit nhị phân
        bits.append(1 if gp >= gc else 0)

    # Bước 3: chia thành nhóm 8-bit, quy đổi sang thập phân
    so_nhom = P // 8          # P=8→1 nhóm, P=16→2 nhóm, P=24→3 nhóm
    gia_tri_nhom = []

    for g in range(so_nhom):
        nhom = bits[g*8 : (g+1)*8]         # lấy 8 bit của nhóm g
        val  = sum(b * (2**i) for i, b in enumerate(nhom))  # bit_0 = LSB = 2^0
        gia_tri_nhom.append(val)

    # Bước 4: lấy MAX trong các nhóm → giá trị LBP cuối cùng
    return max(gia_tri_nhom)


def tinh_lbp_toan_anh(anh_xam, P, R):
    """
    Áp dụng LBP lên toàn bộ ảnh xám bằng vòng lặp.
    Pixel trong vùng biên (cách mép < R pixel) được gán = 0.
    """
    rows, cols = anh_xam.shape
    anh_lbp    = np.zeros((rows, cols), dtype=np.uint8)
    bien       = int(np.ceil(R))   # số hàng/cột biên bỏ qua ở mỗi phía

    for cy in range(bien, rows - bien):
        for cx in range(bien, cols - bien):
            anh_lbp[cy, cx] = tinh_lbp_mot_pixel(anh_xam, cy, cx, P, R)

    return anh_lbp

print('Đã định nghĩa xong các hàm LBP!')

Đã định nghĩa xong các hàm LBP!


---
## Bước 4 – Cấu hình và quét danh sách ảnh

In [3]:
# -------------------------------------------------------
# Đường dẫn thư mục ảnh đầu vào và thư mục lưu kết quả
# -------------------------------------------------------
INPUT_DIR  = os.path.join('anh_xlas', 'anh_xlas')
OUTPUT_DIR = 'ket_qua_lbp'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 5 cấu hình LBP theo yêu cầu đề bài
CAU_HINH = [
    (8,  1, 'P=8,  R=1'),
    (8,  2, 'P=8,  R=2'),
    (16, 2, 'P=16, R=2'),
    (16, 3, 'P=16, R=3'),
    (24, 3, 'P=24, R=3'),
]

# Quét danh sách ảnh trong thư mục đầu vào
EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
ds_anh = sorted([
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(EXTENSIONS)
])

print(f'Tìm thấy {len(ds_anh)} ảnh cần xử lý:')
for ten in ds_anh:
    print(f'  - {ten}')

Tìm thấy 10 ảnh cần xử lý:
  - image_01.jpg
  - image_02.jpg
  - image_03.jpg
  - image_04.jpg
  - image_05.jpg
  - image_06.jpg
  - image_07.jpg
  - image_08.jpg
  - image_09.jpg
  - image_10.jpg


---
## Bước 5 – Xử lý LBP hàng loạt cho 10 ảnh

Với mỗi ảnh, chương trình thực hiện tuần tự:
1. Đọc ảnh → chuyển sang ảnh xám (PIL tự dùng công thức Gray = 0.299R + 0.587G + 0.114B)
2. Áp dụng LBP với cả 5 cấu hình bằng vòng lặp
3. Lưu ảnh kết quả và biểu đồ histogram

In [ ]:
t_bat_dau = time.time()

for idx, ten_anh in enumerate(ds_anh, 1):
    duong_dan = os.path.join(INPUT_DIR, ten_anh)
    print(f'\n[{idx}/{len(ds_anh)}] Đang xử lý: {ten_anh}')

    try:
        # ── 1. Đọc ảnh và chuyển sang ảnh xám ─────────────────────
        img_pil  = Image.open(duong_dan).convert('L')          # chuyển sang xám
        img_xam  = np.array(img_pil, dtype=np.float64)         # ma trận float
        H, W     = img_xam.shape
        print(f'    Kích thước: {H} x {W} pixels')

        ten_goc = os.path.splitext(ten_anh)[0]   # tên file không có đuôi

        # ── 2. Áp dụng LBP với 5 cấu hình ─────────────────────────
        ket_qua = {}
        t0_anh  = time.time()

        for P, R, nhan in CAU_HINH:
            t0 = time.time()
            anh_lbp = tinh_lbp_toan_anh(img_xam, P, R)
            ket_qua[nhan] = anh_lbp
            print(f'    {nhan}: {time.time()-t0:.2f}s  '
                  f'| min={anh_lbp.min():3d}  max={anh_lbp.max():3d}  mean={anh_lbp.mean():.1f}')

        print(f'    => Tổng thời gian: {time.time()-t0_anh:.2f}s')

        # ── 3. Vẽ và lưu ảnh so sánh (6 ô: gốc xám + 5 LBP) ───────
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        fig.suptitle(f'Kết quả LBP – {ten_anh}', fontsize=15, fontweight='bold')
        ax_list = axes.ravel()

        ax_list[0].imshow(img_xam, cmap='gray', vmin=0, vmax=255)
        ax_list[0].set_title('Ảnh xám gốc', fontsize=12, fontweight='bold')
        ax_list[0].axis('off')

        for i, (nhan, anh_lbp) in enumerate(ket_qua.items()):
            ax_list[i+1].imshow(anh_lbp, cmap='gray', vmin=0, vmax=255)
            ax_list[i+1].set_title(nhan, fontsize=12, fontweight='bold')
            ax_list[i+1].axis('off')

        plt.tight_layout()
        path_anh = os.path.join(OUTPUT_DIR, f'{ten_goc}_lbp.png')
        plt.savefig(path_anh, dpi=120, bbox_inches='tight')
        plt.close()

        # ── 4. Vẽ và lưu histogram phân bố LBP ─────────────────────
        fig, axes = plt.subplots(1, 5, figsize=(22, 4))
        fig.suptitle(f'Histogram LBP – {ten_anh}', fontsize=13, fontweight='bold')

        for ax, (nhan, anh_lbp) in zip(axes, ket_qua.items()):
            px = anh_lbp[anh_lbp > 0].ravel()   # bỏ pixel biên = 0
            ax.hist(px, bins=32, range=(0, 255), color='steelblue',
                    edgecolor='white', linewidth=0.5)
            ax.set_title(nhan, fontsize=10, fontweight='bold')
            ax.set_xlabel('Giá trị LBP')
            ax.set_ylabel('Số pixel')
            ax.set_facecolor('#F5F5F5')

        plt.tight_layout()
        path_hist = os.path.join(OUTPUT_DIR, f'{ten_goc}_histogram.png')
        plt.savefig(path_hist, dpi=120, bbox_inches='tight')
        plt.close()

        print(f'    Đã lưu: {path_anh}')
        print(f'    Đã lưu: {path_hist}')

    except Exception as e:
        print(f'    LỖI khi xử lý {ten_anh}: {e}')

print(f'\n Hoàn thành {len(ds_anh)} ảnh sau {time.time()-t_bat_dau:.2f} giây!')
print(f'Kết quả lưu tại thư mục: "{OUTPUT_DIR}/"')


[1/10] Đang xử lý: image_01.jpg
    Kích thước: 1080 x 1920 pixels
